# MMNN vs regular neural network

This notebook compares a regular fully connected neural network (`FCNN`) with the stacked random-feature model (`MMNN`) on harder one-dimensional approximation targets. It is meant to make the model behavior visible: target fit, pointwise error, and summary metrics.

## What to look for

- `FCNN`: all layer weights are trained end to end.
- `MMNN`: each layer samples many frozen nonlinear features, then trains only the linear recombination of those features.
- `oscillatory` and `multiscale` show how models handle high-frequency structure.
- `localized` shows whether models capture narrow bumps.
- `piecewise` shows behavior near a jump discontinuity.

In [1]:
from pathlib import Path
from types import SimpleNamespace
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

# Make imports work whether the notebook is launched from the repo root
# or from the notebooks/ directory.
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "models.py").exists() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from models import count_trainable
from targets import APPROX_TARGETS
from train_approx import choose_device, train_one

torch.set_default_dtype(torch.float32)
plt.rcParams.update({"figure.dpi": 120})

PROJECT_ROOT


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/msun247/anaconda3/envs/mmnn/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/msun247/anaconda3/envs/mmnn/lib/python3.11/site-packages/traitlets/config/application.py", line 1082, in launch_instance
    app.start()
  File "/Users/msun247/anaconda3/envs/mmnn/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 758, in start
    self

PosixPath('/Users/msun247/Downloads/DTB_rep/nn_approx_ladder')

## Experiment configuration

For a fast first run, keep `steps` around 1000 to 2000. Increase it if you want a cleaner comparison after you understand the workflow.

In [2]:
args = SimpleNamespace(
    # Harder target functions than the initial smooth demo.
    targets=["oscillatory", "localized", "multiscale", "piecewise"],
    models=["fcnn", "mmnn"],

    # Training controls.
    steps=1500,
    lr=3e-3,
    batch_size=256,
    n_train=4096,
    n_test=4096,
    n_grid=1200,
    weight_decay=0.0,

    # FCNN capacity.
    tiny_hidden=16,
    hidden=64,
    fcnn_depth=3,
    activation="tanh",

    # MMNN capacity. width controls random features; rank controls the
    # learned state passed between MMNN layers.
    width=192,
    rank=24,
    mmnn_depth=3,
    random_activation="gelu",

    # Reproducibility and output.
    seed=0,
    device="auto",
    out_dir="runs/notebook_mmnn_vs_fcnn",
    verbose=True,
    log_every=500,
)

Path(args.out_dir).mkdir(parents=True, exist_ok=True)
device = choose_device(args.device)
device

device(type='cpu')

## Train FCNN and MMNN

The notebook calls the same `train_one` function used by `train_approx.py`, so command-line and notebook experiments stay aligned.

In [3]:
fitted = {}
results = []

for target_name in args.targets:
    fitted[target_name] = {}
    for model_name in args.models:
        model, result = train_one(target_name, model_name, args, device)
        fitted[target_name][model_name] = model
        results.append(result)
        print(
            f"{target_name:11s} {model_name:5s} "
            f"params={result.params:6d} "
            f"grid_rmse={result.grid_rmse:.3e} "
            f"test_rmse={result.test_rmse:.3e} "
            f"max_grid_err={result.max_abs_error:.3e}"
        )

oscillatory fcnn  step=    1 batch_rmse=5.318e-01
oscillatory fcnn  step=  500 batch_rmse=5.436e-01
oscillatory fcnn  step= 1000 batch_rmse=5.073e-01
oscillatory fcnn  step= 1500 batch_rmse=5.252e-01
oscillatory fcnn  params=  8513 grid_rmse=5.360e-01 test_rmse=5.342e-01 max_grid_err=1.032e+00
oscillatory mmnn  step=    1 batch_rmse=6.135e-01
oscillatory mmnn  step=  500 batch_rmse=5.305e-01
oscillatory mmnn  step= 1000 batch_rmse=3.407e-01
oscillatory mmnn  step= 1500 batch_rmse=1.280e-01
oscillatory mmnn  params=  9457 grid_rmse=1.908e-01 test_rmse=1.940e-01 max_grid_err=7.301e-01
localized   fcnn  step=    1 batch_rmse=3.242e-01
localized   fcnn  step=  500 batch_rmse=1.988e-02
localized   fcnn  step= 1000 batch_rmse=4.837e-03
localized   fcnn  step= 1500 batch_rmse=3.300e-03
localized   fcnn  params=  8513 grid_rmse=2.912e-03 test_rmse=2.918e-03 max_grid_err=7.486e-03
localized   mmnn  step=    1 batch_rmse=4.765e-01
localized   mmnn  step=  500 batch_rmse=4.308e-02
localized   mmn

## Overlay fits and errors

The top panel for each target shows the learned function. The bottom panel shows signed error, so systematic misses are easier to see than in the overlay alone.

In [4]:
x = torch.linspace(-1.0, 1.0, args.n_grid, device=device).unsqueeze(1)
x_cpu = x[:, 0].detach().cpu().numpy()

fig, axes = plt.subplots(len(args.targets), 2, figsize=(11, 3.1 * len(args.targets)), sharex="col")
for row, target_name in enumerate(args.targets):
    target = APPROX_TARGETS[target_name]
    with torch.no_grad():
        y_true = target(x).detach().cpu().numpy()
        predictions = {
            model_name: model(x).detach().cpu().numpy()
            for model_name, model in fitted[target_name].items()
        }

    ax_fit, ax_err = axes[row]
    ax_fit.plot(x_cpu, y_true, color="black", linewidth=2.0, label="target")
    for model_name, pred in predictions.items():
        ax_fit.plot(x_cpu, pred, linewidth=1.5, label=model_name)
        ax_err.plot(x_cpu, pred - y_true, linewidth=1.2, label=model_name)

    ax_fit.set_title(f"{target_name}: function fit")
    ax_fit.set_ylabel("u(x)")
    ax_fit.grid(alpha=0.25)
    ax_fit.legend(fontsize=8)

    ax_err.axhline(0.0, color="black", linewidth=0.8)
    ax_err.set_title(f"{target_name}: signed error")
    ax_err.set_ylabel("prediction - target")
    ax_err.grid(alpha=0.25)

axes[-1, 0].set_xlabel("x")
axes[-1, 1].set_xlabel("x")
fig.tight_layout()

RuntimeError: Numpy is not available

## Metric comparison

Lower is better. The y-axis is logarithmic so a 10x difference is visible.

In [ ]:
targets = list(dict.fromkeys(r.target for r in results))
models = list(dict.fromkeys(r.model for r in results))
xpos = np.arange(len(targets))
bar_width = min(0.8 / len(models), 0.3)
offsets = (np.arange(len(models)) - (len(models) - 1) / 2.0) * bar_width

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharex=True)
for ax, metric_name, title in [
    (axes[0], "grid_rmse", "Grid RMSE"),
    (axes[1], "test_rmse", "Random test RMSE"),
    (axes[2], "max_abs_error", "Max grid error"),
]:
    for model_name, offset in zip(models, offsets):
        values = [
            getattr(next(r for r in results if r.target == target_name and r.model == model_name), metric_name)
            for target_name in targets
        ]
        ax.bar(xpos + offset, values, width=bar_width, label=model_name)
    ax.set_title(title)
    ax.set_yscale("log")
    ax.set_xticks(xpos, targets, rotation=25, ha="right")
    ax.grid(axis="y", alpha=0.25)

axes[-1].legend(fontsize=8)
fig.tight_layout()

## Parameter counts

MMNN can have a large random feature width while still training relatively few parameters, because the random feature projections are frozen buffers.

In [ ]:
for target_name in args.targets[:1]:
    for model_name in args.models:
        model = fitted[target_name][model_name]
        total_tensors = sum(p.numel() for p in model.state_dict().values())
        print(
            f"{model_name:5s}: trainable={count_trainable(model):6d}, "
            f"state_dict tensors including frozen buffers={total_tensors:6d}"
        )

## Things to try next

- Increase `steps` to see whether FCNN catches up on oscillatory targets.
- Increase MMNN `width` to add more frozen nonlinear features.
- Increase MMNN `rank` to let more information pass between stacked layers.
- Switch `random_activation` between `gelu`, `tanh`, and `sin`.